# ISIC 2019 Classification Training with Comet Logging

This notebook trains a custom skin-lesion classifier using the ISIC 2019 dataset (multi-class).



In [ ]:
from io import BytesIO
from pathlib import Path
import os
import random
import shutil

import comet_ml
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from sklearn.metrics import classification_report
from ultralytics import YOLO



## 2. Dataset Setup

Download ISIC 2019 data and place it locally. Expected raw layout:

**Training**
- `ISIC_2019_Training_Input/` (images)
- `ISIC_2019_Training_GroundTruth.csv` (labels)

**Test**
- `ISIC_2019_Test_Input/` (images)
- `ISIC_2019_Test_GroundTruth.csv` (labels)

Set `RAW_DATA_DIR` to the folder containing those files.

### YOLO Classification Folder Format

Ultralytics YOLO expects classification datasets as folders split by `train/`, `val/`, and `test/`, with one subfolder per class:

```
OUTPUT_DIR/
├── train/
│   ├── AK/
│   ├── BCC/
│   └── ...
├── val/
│   ├── AK/
│   ├── BCC/
│   └── ...
└── test/
    ├── AK/
    ├── BCC/
    └── ...
```

The prep cell below converts the ISIC CSV labels into this folder structure for you.


In [ ]:
RAW_DATA_DIR = Path("/home/jrterven/Documents/data/isic2019_raw")
OUTPUT_DIR = Path("/home/jrterven/Documents/data/isic2019_yolo")
SPLIT_SEED = 42
VAL_FRACTION = 0.1
MAX_SAMPLES_PER_CLASS = None  # e.g., 200 for a quick demo
EXCLUDED_LABELS = {"UNK", "unknown", "background"}
EXCLUDED_LABELS_LOWER = {label.lower() for label in EXCLUDED_LABELS}

In [ ]:

def safe_link_or_copy(src: Path, dst: Path) -> None:
    """Create a symlink (or copy) from src to dst, ensuring dst's parent exists."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def is_excluded_label(label: str) -> bool:
    """Return True if the label should be ignored (e.g., UNK)."""
    return label.strip().lower() in EXCLUDED_LABELS_LOWER


def drop_unknown_rows(df: pd.DataFrame, class_columns: list[str], split_name: str) -> pd.DataFrame:
    """Drop rows without any known label after filtering class columns."""
    row_sum = df[class_columns].sum(axis=1)
    dropped = int((row_sum == 0).sum())
    if dropped:
        print(f"Dropping {dropped} {split_name} rows without known labels")
    return df.loc[row_sum > 0].copy()


def load_isic_2019_train():
    """Load training images and labels from the ISIC 2019 training CSV."""
    images_dir = RAW_DATA_DIR / "ISIC_2019_Training_Input"
    labels_csv = RAW_DATA_DIR / "ISIC_2019_Training_GroundTruth.csv"
    if not labels_csv.exists():
        raise FileNotFoundError(f"Missing CSV: {labels_csv}")
    df = pd.read_csv(labels_csv)
    class_columns = [col for col in df.columns if col != "image" and not is_excluded_label(col)]
    df = drop_unknown_rows(df, class_columns, "train")
    df["label"] = df[class_columns].idxmax(axis=1)
    df["filename"] = df["image"] + ".jpg"
    return images_dir, df[["filename", "label"]], class_columns


def align_label_columns(df: pd.DataFrame, class_columns: list[str]) -> pd.DataFrame:
    """Align label columns to match the training set class list."""
    extra_columns = [col for col in df.columns if col not in class_columns and col != "image"]
    if extra_columns:
        print(f"Dropping extra label columns in test set: {extra_columns}")
    for column in class_columns:
        if column not in df.columns:
            df[column] = 0
    return df[["image"] + class_columns]


def load_isic_2019_test(class_columns: list[str]):
    """Load test images and labels, aligned to the training class list."""
    images_dir = RAW_DATA_DIR / "ISIC_2019_Test_Input"
    labels_csv = RAW_DATA_DIR / "ISIC_2019_Test_GroundTruth.csv"
    if not labels_csv.exists():
        raise FileNotFoundError(f"Missing CSV: {labels_csv}")
    df = pd.read_csv(labels_csv)
    df = align_label_columns(df, class_columns)
    df = drop_unknown_rows(df, class_columns, "test")
    df["label"] = df[class_columns].idxmax(axis=1)
    df["filename"] = df["image"] + ".jpg"
    return images_dir, df[["filename", "label"]]


def sample_per_class(df, max_samples):
    """Optionally subsample each class to a maximum number of samples."""
    if max_samples is None:
        return df
    return (
        df.groupby("label", group_keys=False)
        .apply(lambda group: group.sample(min(len(group), max_samples), random_state=SPLIT_SEED))
        .reset_index(drop=True)
    )


train_images_dir, train_df, class_columns = load_isic_2019_train()
test_images_dir, test_df = load_isic_2019_test(class_columns)

train_df = sample_per_class(train_df, MAX_SAMPLES_PER_CLASS)
train_df, val_df = train_test_split(
    train_df, test_size=VAL_FRACTION, random_state=SPLIT_SEED, stratify=train_df["label"]
)


def write_split(split_df, split_name, images_dir):
    """Write a dataset split to the YOLO folder structure."""
    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc=f"{split_name} split"):
        src = images_dir / row["filename"]
        if not src.exists():
            raise FileNotFoundError(f"Missing image: {src}")
        dst = OUTPUT_DIR / split_name / row["label"] / row["filename"]
        safe_link_or_copy(src, dst)


if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_split(train_df, "train", train_images_dir)
write_split(val_df, "val", train_images_dir)
write_split(test_df, "test", test_images_dir)

print("Dataset prepared at", OUTPUT_DIR.resolve())




In [ ]:
# Show the sizes of each split
print("\nSplit sizes:")
for split in ["train", "val", "test"]:
    split_dir = OUTPUT_DIR / split
    print(f"{split}: {len(list(split_dir.glob('**/*')))}")

In [ ]:
# Per-class training distribution (reading from disk)
train_dir = OUTPUT_DIR / "train"
class_counts_dict = {}

if train_dir.exists():
    for class_folder in sorted(train_dir.iterdir()):
        if class_folder.is_dir():
            # Count images in each class folder
            count = len(list(class_folder.glob("*")))
            class_counts_dict[class_folder.name] = count

train_counts = pd.Series(class_counts_dict)
train_distribution = pd.DataFrame(
    {
        "count": train_counts,
        "percent": (train_counts / train_counts.sum() * 100).round(2),
    }
)

display(train_distribution)

ax = train_counts.plot(kind="bar", figsize=(8, 4), color="steelblue")
ax.set_title("Training Split Class Distribution")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

## Setup Comet Logging


In [ ]:
PROJECT_NAME = "ISIC_classification3"
MODEL_NAME = "yolo26l-cls"
EXPERIMENT_NAME = "yolov26l_200ep"

comet_ml.login(project_name=PROJECT_NAME)
os.environ["COMET_EXPERIMENT_NAME"] = EXPERIMENT_NAME  # fixed name

## 3. Train the Classifier

If you do not have a GPU, set `device='cpu'` and reduce the batch size.

### Output Structure

Training outputs are saved under the `runs/` directory (relative to the notebook):

```
runs/
└── isic2019_cls/
    ├── weights/
    │   ├── best.pt
    │   └── last.pt
    ├── results.csv
    └── ...
```

Use `best.pt` for evaluation and inference, or change the run name via `name=...` in the train call.


In [ ]:
# Train the model - Comet will automatically log metrics
model = YOLO(f"{MODEL_NAME}.pt")

results = model.train(
    data=str(OUTPUT_DIR),
    epochs=200,
    patience=20,
    imgsz=224,
    batch=128,
    device=[0,1], # "cpu", "0"
    project=PROJECT_NAME,
    name=EXPERIMENT_NAME  # Local save directory name
)

## 4. Evaluate on the Test Split + Quick Inference

Use the test split created from the labeled data (so ground truth is available). For a CLI-only workflow, you can also run:

```bash
python scripts/classify/test_isic2019.py --model runs/isic2019_cls/weights/best.pt --data data/isic2019_yolo
```


In [ ]:
# load trained model
#model = YOLO("runs/isic2019_cls_cuda/weights/best.pt")

In [ ]:
metrics = model.val(
    data=str(OUTPUT_DIR),
    split="test",
    project=PROJECT_NAME,
    name=f"{EXPERIMENT_NAME}_test"
)
print(metrics)

# Classification report (scikit-learn)
image_paths = [
    path
    for path in (OUTPUT_DIR / "test").rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]
class_labels = sorted({path.parent.name for path in image_paths})

y_true = []
y_pred = []
for image_path in tqdm(image_paths, desc="Computing Classification Report"):
    y_true.append(image_path.parent.name)
    preds = model.predict(source=str(image_path), verbose=False)
    top1 = int(preds[0].probs.top1)
    y_pred.append(preds[0].names[top1])

report = classification_report(y_true, y_pred, labels=class_labels, target_names=class_labels)
print(report)

report_path = Path(metrics.save_dir) / "classification_report.txt"
report_path.write_text(report)
print(f"Saved classification report to: {report_path}")

In [ ]:
# Log extra metrics to Comet (attach to existing experiment)
# Paste the experiment KEY from Comet
EXPERIMENT_KEY = "16de8c45faee4fe89b56888d67ef645e"
if EXPERIMENT_KEY:
    experiment = comet_ml.ExistingExperiment(previous_experiment=EXPERIMENT_KEY)
else:
    raise RuntimeError(
        "No active Comet experiment found. Set COMET_EXPERIMENT_KEY."
    )

print("Experiment:", experiment)
report_dict = classification_report(
    y_true,
    y_pred,
    labels=class_labels,
    target_names=class_labels,
    output_dict=True,
)

experiment.log_metrics(
    {
        "precision_macro": report_dict["macro avg"]["precision"],
        "recall_macro": report_dict["macro avg"]["recall"],
        "f1_macro": report_dict["macro avg"]["f1-score"],
        "precision_weighted": report_dict["weighted avg"]["precision"],
        "recall_weighted": report_dict["weighted avg"]["recall"],
        "f1_weighted": report_dict["weighted avg"]["f1-score"],
        "accuracy": report_dict["accuracy"],
    }
)

for label in class_labels:
    metrics = report_dict[label]
    experiment.log_metrics(
        {
            f"{label}/precision": metrics["precision"],
            f"{label}/recall": metrics["recall"],
            f"{label}/f1": metrics["f1-score"],
            f"{label}/support": metrics["support"],
        }
    )

experiment.end()

## 5. Random Test Image + Probabilities

Run this cell to sample a new test image, display it, and print class probabilities.

In [ ]:
# Pick a random test image each time
random_image = random.choice(list((OUTPUT_DIR / "test").rglob("*.jpg")))

# Display the image (adjust display_width to control size)
image = Image.open(random_image)
display_width = 400
scale = display_width / image.width
image_display = image.resize((display_width, int(image.height * scale)))
display(image_display)

# Run prediction
preds = model.predict(source=str(random_image), verbose=False)
probs = preds[0].probs

if probs is None:
    raise ValueError("No probabilities returned. Ensure you are using a classification model.")

# Map class indices to names
class_names = preds[0].names
scores = probs.data.cpu().numpy()

full_class_names = {
    "AK":  "Actinic keratoses",
    "BCC": "Basal cell carcinoma",
    "BKL": "Benign keratosis-like lesions",
    "DF":  "Dermatofibroma",
    "MEL": "Melanoma",
    "NV":  "Melanocytic nevi",
    "SCC": "Squamous cell carcinoma",
    "VASC":"Vascular lesions",
}

ground_truth_code = random_image.parent.name
ground_truth_name = full_class_names.get(ground_truth_code, ground_truth_code)
print(f"Ground truth: {ground_truth_name} ({ground_truth_code})\n")

sorted_items = sorted(enumerate(scores), key=lambda item: item[1], reverse=True)
labels = []
values = []

print("Prediction probabilities:")
for class_id, score in sorted_items:
    class_code = class_names.get(class_id, str(class_id))
    class_name = full_class_names.get(class_code, class_code)
    labels.append(f"{class_name} ({class_code})")
    values.append(score)
    print(f"  {class_name} ({class_code}): {score:.4f}")

# Horizontal bar chart of probabilities (render as image to ensure display)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(labels, values, color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Probability")
ax.set_title("Prediction Probabilities")
fig.tight_layout()

buffer = BytesIO()
fig.savefig(buffer, format="png", dpi=150)
plt.close(fig)
buffer.seek(0)
display(Image.open(buffer))